# 01 · Ingestão e qualidade dos dados

**O problema que ninguém esperava:** os dados não vieram como CSV. Vieram como
`Diabetes-2026.csv.pdf` — **109 MB, 4.374 páginas** de tabela renderizada.

Extrair por ordem de leitura de texto é frágil: a especificação PDF **não garante**
que a ordem dos tokens corresponda à ordem visual. A ingestão reconstrói as linhas
**por coordenada de bounding box**.

> Documento completo: [`docs/01-diagnostico-dos-dados.md`](../docs/01-diagnostico-dos-dados.md)


In [1]:
import sys, json
from pathlib import Path

# a raiz e onde existe src/ — funciona rodando de notebooks/ ou da raiz do repo
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

GOLD = RAIZ / "data" / "processed" / "gold"
def ler(nome, base=GOLD):
    return json.loads((base / nome).read_text(encoding="utf-8"))


## A extração — e a prova de que funcionou


In [2]:
manifesto = ler("_manifest_ingestao.json", RAIZ / "data" / "raw")["manifest"]
pd.Series({
    "páginas processadas":  manifesto["paginas"],
    "linhas reconstruídas": manifesto["n_linhas"],
    "colunas":              manifesto["n_colunas"],
    "linhas em quarentena": manifesto["n_quarentena"],
    "segundos":             manifesto["segundos"],
    "sha256 do CSV":        manifesto["sha256_csv"][:16] + "...",
}).to_frame("valor")


,valor
páginas processadas,4374
linhas reconstruídas,253680
colunas,22
linhas em quarentena,0
segundos,48.6
sha256 do CSV,e2b5c90b37b68f8d...


**253.680 linhas, 22 colunas, zero em quarentena** — bate exatamente com o enunciado.

Nenhuma linha foi descartada em silêncio: cardinalidade e parseabilidade são
validadas linha a linha, e o que falha vai para quarentena com página e motivo.


## Limpeza — 7 regras, cada uma com contagem


In [3]:
rel = ler("_relatorio_limpeza.json", RAIZ / "data" / "processed")
pd.Series({
    "linhas de entrada":        rel["entrada_linhas"],
    "violações de domínio":     len(rel["violacoes_dominio"]),
    "linhas em quarentena":     rel["linhas_quarentena"],
    "duplicatas exatas":        rel["duplicatas_exatas"],
    "grupos com alvo conflitante": rel["grupos_alvo_conflitante"],
    "IMC > 60 (marcado, não removido)": rel["imc_extremo"],
    "memória final (MB)":       rel["memoria_mb"],
}).to_frame("valor")


,valor
linhas de entrada,253680.00
violações de domínio,0.00
linhas em quarentena,0.00
duplicatas exatas,23899.00
grupos com alvo conflitante,1834.00
"IMC > 60 (marcado, não removido)",805.00
memória final (MB),8.63


### Distribuição do alvo — por que acurácia foi banida


In [4]:
alvo = pd.Series(rel["distribuicao_alvo_pct"])
alvo.index = ["0 · sem diabetes", "1 · pré-diabetes", "2 · diabetes"]
display(alvo.to_frame("% da amostra"))
print(f"\nUm classificador que sempre responde '0' acerta {alvo.iloc[0]:.1f}%.")
print("Por isso a métrica principal é PR-AUC, não acurácia (ADR 0005).")


,% da amostra
0 · sem diabetes,84.241
1 · pré-diabetes,1.826
2 · diabetes,13.933



Um classificador que sempre responde '0' acerta 84.2%.
Por isso a métrica principal é PR-AUC, não acurácia (ADR 0005).


## Os quatro problemas que definem o trabalho

1. **23.899 duplicatas exatas** (9,4%) → risco de vazamento treino/teste
2. **1.834 grupos com rótulo contraditório** → teto de Bayes mensurável
3. **Desbalanceamento severo** — a classe pré-diabetes tem 1,8%
4. **Zero códigos 77/99 de renda** → a amostra foi **truncada** antes de chegar

O item 4 é o que abre o resto do projeto: quem não declarou renda foi **excluído**,
e isso é medível comparando com a fonte original — o notebook 02.
